## Notebook Overview

This notebook evaluates the performance of the trained Gradient Boosting model on unseen data. It computes accuracy metrics, visualizes prediction quality, and analyzes model behavior.

It will further explore edge cases and the model's behaviour, identifying any potential flaws or insufficiencies. 

## Imports and setup

In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import sys
sys.path.append('../backend')

from model_loader import load_model
from preprocessing import preprocess_input

c:\Users\CLL\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.6.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## Load Model & Evaluation Data

For evaluation purposes, we will be using `models\V1_GradientBoosting_F1_Race_Predictor_model.joblib` as it is trained on 2015 to 2024 race data, ensures unbiased predictions for 2025 races.

In [2]:
model = load_model('../models/V1_GradientBoosting_F1_Race_Predictor_model.joblib', True)

model

✅ Model loaded successfully from: ../models/V1_GradientBoosting_F1_Race_Predictor_model.joblib


c:\Users\CLL\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DummyRegressor from version 1.6.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\CLL\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.6.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\CLL\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estima

,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.05
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",300
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are""friedman_mse"" for the mean squared error with improvement score byFriedman, ""squared_error"" for mean squared error. The default value of""friedman_mse"" is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft

## Preprocess the Evaluation Data

Since the `models\V1_GradientBoosting_F1_Race_Predictor_model.joblib` model is trained on race & qualifying data from 2015 to 2024. To predict the results of the 2025 races, we will use the corresponding qualifying data for 2025.

You can modify the `year` and `round` variables below as needed. However, please keep in mind that there might be some bias when applying this model to predict outcomes for races in 2025, as the model was trained on data from 2015 to 2024.

In [3]:
year = 2025
round = 12

df = pd.read_csv(r'C:\Users\CLL\OneDrive\Documents\GitHub\F1-Race-Predictor\notebooks\2015_to_2025_df.csv')
eval_df = df.loc[(df['Season'] == year) & (df['Round'] == round)]

eval_df.head(5)

,Season,Round,EventName,Abbreviation,TeamName,GridPosition,Q1_s,Q2_s,Q3_s,Position
4438,2025,12,British Grand Prix,NOR,McLaren,3.0,86.123,85.231,85.010,1.0
4439,2025,12,British Grand Prix,PIA,McLaren,2.0,85.963,85.316,84.995,2.0
4440,2025,12,British Grand Prix,HUL,Kick Sauber,19.0,86.574,9999.000,9999.000,3.0
4441,2025,12,British Grand Prix,HAM,Ferrari,5.0,86.296,85.084,85.095,4.0
4442,2025,12,British Grand Prix,VER,Red Bull Racing,1.0,85.886,85.316,84.892,5.0


`notebooks\2015_to_2025_df.csv` is updated after any previously missing data are replaced and cleaned, we now call `preprocess_input()` imported from `backend\preprocessing.py` to prepare the data for use by our model.

In [4]:
test = preprocess_input(eval_df)

X = test.drop('Position', axis=1)
y = test['Position']

test.head(5) # For visualisation purposes

,Season,Round,GridPosition,Q1_s,Q2_s,Q3_s,Position,Abbreviation_ALB,Abbreviation_ALO,Abbreviation_ANT,...,EventName_Russian Grand Prix,EventName_Sakhir Grand Prix,EventName_Saudi Arabian Grand Prix,EventName_Singapore Grand Prix,EventName_Spanish Grand Prix,EventName_Styrian Grand Prix,EventName_São Paulo Grand Prix,EventName_Turkish Grand Prix,EventName_Tuscan Grand Prix,EventName_United States Grand Prix
4438,2025,12,3.0,86.123,85.231,85.010,1.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4439,2025,12,2.0,85.963,85.316,84.995,2.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4440,2025,12,19.0,86.574,9999.000,9999.000,3.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4441,2025,12,5.0,86.296,85.084,85.095,4.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4442,2025,12,1.0,85.886,85.316,84.892,5.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Make Predictions

In [5]:
predictions = model.predict(X)

predictions

array([ 3.8991749 ,  3.46459521, 13.81606482,  6.79421738,  3.6642619 ,
       10.57729497, 14.32563559, 13.79290551,  8.70507937,  6.10202725,
       12.91079615, 12.24141125, 13.25073362,  7.15759138, 10.88927994,
       14.36505515, 13.79435156, 15.10137476, 15.1488419 , 15.2667469 ])

## Evaluate Performance

In [6]:
eval_df['ExactPredictedPosition'] = predictions

df_sorted = eval_df.sort_values('ExactPredictedPosition')
df_sorted['PredictedPosition'] = [float(i) for i in range (1, 21)]
df_sorted = df_sorted.sort_values('Position')

df_sorted[['Season', 'Round', 'EventName', 'Abbreviation', 'TeamName', 'GridPosition', 'Position', 'PredictedPosition', 'ExactPredictedPosition']]

C:\Users\CLL\AppData\Local\Temp\ipykernel_28120\2258667622.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eval_df['ExactPredictedPosition'] = predictions


,Season,Round,EventName,Abbreviation,TeamName,GridPosition,Position,PredictedPosition,ExactPredictedPosition
4438,2025,12,British Grand Prix,NOR,McLaren,3.0,1.0,3.0,3.899175
4439,2025,12,British Grand Prix,PIA,McLaren,2.0,2.0,1.0,3.464595
4440,2025,12,British Grand Prix,HUL,Kick Sauber,19.0,3.0,15.0,13.816065
4441,2025,12,British Grand Prix,HAM,Ferrari,5.0,4.0,5.0,6.794217
4442,2025,12,British Grand Prix,VER,Red Bull Racing,1.0,5.0,2.0,3.664262
4443,2025,12,British Grand Prix,GAS,Alpine,8.0,6.0,8.0,10.577295
4444,2025,12,British Grand Prix,STR,Aston Martin,17.0,7.0,16.0,14.325636
4445,2025,12,British Grand Prix,ALB,Williams,13.0,8.0,13.0,13.792906
4446,2025,12,British Grand Prix,ALO,Aston Martin,7.0,9.0,7.0,8.705079
4447,2025,12,British Grand Prix,RUS,Mercedes,4.0,10.0,4.0,6.102027
